# ML-04 — Search Intelligence Data Contract

**Lane:** Search visibility / content decline detection  
**Feature window:** February 2026  
**Decision cutoff:** 2026-02-28  
**Label window:** March 2026  
**Development month:** March 2026 (mid-panel; June 2026 remains sealed)

This notebook defines the data contract, verifies the warehouse grain/count/availability, builds exactly five decision-time features, and demonstrates one deliberate label-leakage trap.

> **Important:** Run the query cells against the gated FlyRank warehouse. The outputs are intentionally generated from the real warehouse rather than hard-coded.


## 1) Unit of analysis + time window

**What one row means:** One row in the modeling frame represents one pseudonymized **client + content item** after aggregating the daily search-performance fact table over the February 2026 feature window.

**Feature window:** 2026-02-01 through 2026-02-28. Only information available by the decision cutoff (2026-02-28) is used as a model feature.

**Decision:** At the end of February, rank content items that may lose search visibility in the following month.

**Label window:** 2026-03-01 through 2026-03-31. The outcome is `went_dark = 1` when measured March search clicks are zero, subject to the minimum-impression/data-availability rule below.

**Why this is a time-safe contract:** February is the information available when the decision is made; March is the future outcome window. June 2026 is kept sealed and is not used to develop the label logic.


In [7]:
# Setup: install/import dependencies and connect to the gated FlyRank warehouse.
%pip install -q duckdb huggingface_hub pandas scikit-learn

import duckdb
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN is missing. Add it in Colab Secrets and enable Notebook access."

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])
con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN getvariable('hf_token')
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
DIM = f"{REL}/dim_content.parquet"
CLI = f"{REL}/dim_clients.parquet"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("connected; feature window = Feb 2026; label window = Mar 2026")


connected; feature window = Feb 2026; label window = Mar 2026


### Contract buckets

**Feature fields (exactly five):**
1. `impressions_feb` — February Google Search impressions.
2. `clicks_feb` — February Google Search clicks.
3. `ctr_feb` — February clicks divided by impressions.
4. `avg_position_feb` — February impression-weighted average position.
5. `measured_days_feb` — number of February days with measured Search Console data.

**Label:** `went_dark` — March has zero measured Google Search clicks, with the minimum-impression/availability rule applied.

**Context:** pseudonymized client/content identifiers are used to group and validate rows, but are not predictive features.

**Deliberately excluded:** March outcome fields, including March clicks/impressions and `trend_direction`/`trend_pct`. They are unavailable at the February decision moment and would leak the future label. IDs are also excluded from model inputs because they identify entities rather than transferable search signals.


## 3) Verify it with queries

The next **three cells are the three verification queries required by the assignment**.

- Query 1 verifies the daily fact grain.
- Query 2 verifies the March development slice's row count and date span.
- Query 3 verifies availability using `IS TRUE`.


In [8]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

try:
    info = api.dataset_info("FlyRank/internship-warehouse")
    print("✅ Hugging Face dataset is accessible")
    print("Dataset:", info.id)
except Exception as e:
    print("❌ Dataset access failed")
    print(type(e).__name__)
    print(e)

✅ Hugging Face dataset is accessible
Dataset: FlyRank/internship-warehouse


In [9]:
# Verification query 2 — March development slice row count and date span
q2 = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {MAR}
""")
q2


HTTPException: HTTP Error: HTTP GET error on 'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance/month=2026-03/data_0.parquet' (HTTP 403)

In [ ]:
# Verification query 3 — availability, explicitly using IS TRUE
q3 = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) * 1.0 / COUNT(*) AS availability_rate
FROM {MAR}
""")
q3


### Five-feature frame

Each feature is constructed only from the February feature window and is therefore knowable at the 2026-02-28 decision cutoff.

| Feature | Available when? |
|---|---|
| `impressions_feb` | Knowable by the cutoff because it uses only February Search Console impressions. |
| `clicks_feb` | Knowable by the cutoff because it uses only February Search Console clicks. |
| `ctr_feb` | Knowable by the cutoff because it is calculated from February clicks and impressions. |
| `avg_position_feb` | Knowable by the cutoff because it is calculated from February position sums and impressions. |
| `measured_days_feb` | Knowable by the cutoff because it counts February days with `gsc_data_available IS TRUE`. |


In [ ]:
# Build the five-feature February frame.
# The position feature is impression-weighted, not a simple average of daily averages.
feature_frame = con.sql(f"""
WITH feb AS (
    SELECT
        client_id,
        content_id,
        SUM(gsc_impressions) AS impressions_feb,
        SUM(gsc_clicks) AS clicks_feb,
        SUM(gsc_sum_position) AS sum_position_feb,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS measured_days_feb
    FROM {FEB}
    GROUP BY client_id, content_id
)
SELECT
    client_id,
    content_id,
    impressions_feb,
    clicks_feb,
    clicks_feb * 1.0 / NULLIF(impressions_feb, 0) AS ctr_feb,
    sum_position_feb * 1.0 / NULLIF(impressions_feb, 0) AS avg_position_feb,
    measured_days_feb
FROM feb
WHERE impressions_feb > 0
""").df()

feature_frame.head(10)


## Label and deliberate leakage trap

The future label is built from March, not February. For the leakage demonstration, we deliberately add `clicks_mar` — a direct source of the March outcome — to the feature set.

That feature is **not allowed** in the honest model. If it produces a near-perfect/perfect score, that is the trap: the model has been handed the answer.

The honest frame below removes the leaked column again.


In [ ]:
# Build the March label with the same client/content grain.
label_frame = con.sql(f"""
WITH mar AS (
    SELECT
        client_id,
        content_id,
        SUM(gsc_impressions) AS impressions_mar,
        SUM(gsc_clicks) AS clicks_mar,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS measured_days_mar
    FROM {MAR}
    GROUP BY client_id, content_id
)
SELECT
    client_id,
    content_id,
    impressions_mar,
    clicks_mar,
    measured_days_mar,
    CASE
        WHEN measured_days_mar > 0
         AND impressions_mar >= 100
         AND clicks_mar = 0
        THEN 1 ELSE 0
    END AS went_dark
FROM mar
""").df()

frame = feature_frame.merge(
    label_frame[["client_id", "content_id", "impressions_mar", "clicks_mar", "measured_days_mar", "went_dark"]],
    on=["client_id", "content_id"],
    how="inner",
)

print("Frame rows:", len(frame))
print("Positive labels:", int(frame["went_dark"].sum()))
print("Positive rate:", round(frame["went_dark"].mean() * 100, 2), "%")
frame.head()


In [ ]:
# Deliberate leakage experiment: add the future March clicks as a feature.
# A single feature that directly contains the outcome information should make the score suspiciously high.
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

honest_cols = [
    "impressions_feb",
    "clicks_feb",
    "ctr_feb",
    "avg_position_feb",
    "measured_days_feb",
]

leaky_cols = honest_cols + ["clicks_mar"]

model_df = frame[honest_cols + ["clicks_mar", "went_dark"]].dropna()

X_train, X_test, y_train, y_test = train_test_split(
    model_df[leaky_cols],
    model_df["went_dark"],
    test_size=0.20,
    random_state=42,
    stratify=model_df["went_dark"],
)

leaky_model = DecisionTreeClassifier(max_depth=3, random_state=42)
leaky_model.fit(X_train, y_train)
leaky_pred = leaky_model.predict(X_test)

leaky_accuracy = accuracy_score(y_test, leaky_pred)
print(f"Leakage experiment accuracy: {leaky_accuracy:.3f}")
print("The March clicks column is intentionally leaked and must be removed.")


In [ ]:
# Honest model: remove the future label-derived column.
X_train, X_test, y_train, y_test = train_test_split(
    model_df[honest_cols],
    model_df["went_dark"],
    test_size=0.20,
    random_state=42,
    stratify=model_df["went_dark"],
)

honest_model = DecisionTreeClassifier(max_depth=3, random_state=42)
honest_model.fit(X_train, y_train)
honest_pred = honest_model.predict(X_test)

honest_accuracy = accuracy_score(y_test, honest_pred)
print(f"Honest accuracy after removing leakage: {honest_accuracy:.3f}")
print("Kept features:", honest_cols)


## 4) Data limits

**Named limitation — Search Console measurement/availability:** the warehouse does not guarantee a measured Search Console observation for every client/content item on every day. The contract therefore counts only rows where `gsc_data_available IS TRUE`, and the March label requires measured data and at least 100 impressions. This avoids treating missing measurement as genuine zero traffic, but it can exclude some pages from the training frame and may reduce representativeness.

A second limitation is that the label is a proxy for a future search outcome: `went_dark` means zero measured clicks under the stated threshold, not proof that a page is intrinsically poor or that its content caused the decline.


## 5) Self-check

- [x] Plain-words data contract completed.
- [x] Feature window, decision cutoff, and future label window stated.
- [x] Exactly three verification queries shown.
- [x] Verification includes an explicit `IS TRUE` availability check.
- [x] Five features maximum, each available at decision time.
- [x] Deliberate label-derived leakage experiment shown.
- [x] Leaked feature removed for the honest result.
- [x] One named data limitation documented.
- [ ] Run all cells successfully and commit the executed notebook to `work/notebooks/w03_data_contract.ipynb`.

**Final submission:** submit the URL of your GitHub repository, not a separate notebook URL.
